# Woche 8 · Tool Use: Das Modell ruft eure Funktionen auf
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Bisher kann der Assistent nur lesen und schreiben. Heute bekommt er **Werkzeuge**: Er kann den Maschinenstatus abfragen, die Wartungshistorie ansehen, im Handbuch suchen und Tickets anlegen.

**Die wichtigste Einsicht vorweg:** Das Modell führt nichts selbst aus. Es **schlägt einen Aufruf vor** (`tool_calls`), **euer Code** führt ihn aus und schickt das Ergebnis zurück (Nachricht mit der Rolle `tool`). Ihr behaltet die Kontrolle.

**Lernziele**
- Den Ablauf aus Tool-Definition, Tool-Aufruf, Ausführung, Tool-Ergebnis und Antwort erklären
- Tools mit JSON-Schema und guten Beschreibungen definieren
- Fehler aus Tools sauber zurückmelden
- Schreibende Tools mit einer Bestätigung absichern
- Das RAG aus Woche 7 als Tool kapseln

**Aufbau:** A Ein Tool von Hand · B Mehr Tools, eine Schleife · C Schreibende Tools absichern · D Ausprobieren und messen
Pflicht: TODO 1–6, Ziel: TODO 7–8. Toolbox-Schlüssel nötig, etwa 40 Anfragen pro Person.

In [ ]:
%pip install -q openai==3.24.0 rank-bm25==0.2.2

In [ ]:
import json, re, datetime
from openai import OpenAI
from rank_bm25 import BM25Okapi

def check(name, test):
    """Selbsttest: zeigt ✅ oder ❌. Fehler im Test gelten als 'noch nicht gelöst'."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    import os
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
client = OpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1", max_retries=5)
MODEL = "kit.mistral-small-4-119b-a8b"

def antworttext(antwort):
    return antwort.choices[0].message.content or ""

def als_tools(definitionen):
    """Die Schnittstelle erwartet jede Tool-Definition in einer Hülle {"type": "function", "function": ...}."""
    return [{"type": "function", "function": d} for d in definitionen]

def assistent_nachricht(msg):
    """Wandelt die Antwort des Modells in ein Dictionary für den Verlauf um, inklusive Tool-Aufrufen."""
    d = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        d["tool_calls"] = [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in msg.tool_calls]
    return d

def zeige_nachrichten(nachrichten):
    """Kompakte Ausgabe einer Nachrichtenliste."""
    for n in nachrichten:
        if n["role"] == "tool":
            fehler = " (Fehler)" if str(n["content"]).startswith("FEHLER") else ""
            print(f"{'tool':9s} | Ergebnis{fehler}: {str(n['content'])[:80]}  id={n['tool_call_id'][:12]}…")
            continue
        if n.get("content"):
            print(f"{n['role']:9s} | text: {str(n['content'])[:90]}")
        for tc in n.get("tool_calls", []):
            print(f"{n['role']:9s} | tool_call: {tc['function']['name']}({tc['function']['arguments']})  id={tc['id'][:12]}…")
print("Schlüssel geladen:", bool(API_KEY))

**Daten der Nordwerk GmbH** (erfunden, wie im Startpaket B): Maschinen, Wartungshistorie, Störungstickets und das Wartungshandbuch. Einfach ausführen.

In [ ]:
MASCHINEN = [
    {'id': 'M-01', 'name': 'CNC-Fräszentrum 1', 'typ': 'Kessler KFZ 800', 'baujahr': 2018, 'standort': 'Halle 1, Platz 4', 'status': 'in Betrieb', 'betriebsstunden': 28450, 'letzte_wartung': '2026-08-12', 'service': 'Kessler Service, SV-1180'},
    {'id': 'M-02', 'name': 'CNC-Fräszentrum 2', 'typ': 'Kessler KFZ 800 plus', 'baujahr': 2021, 'standort': 'Halle 1, Platz 5', 'status': 'in Betrieb', 'betriebsstunden': 14120, 'letzte_wartung': '2026-09-02', 'service': 'Kessler Service, SV-1180'},
    {'id': 'M-03', 'name': 'CNC-Drehmaschine', 'typ': 'Hartwig TD 400', 'baujahr': 2015, 'standort': 'Halle 1, Platz 9', 'status': 'eingeschränkt', 'betriebsstunden': 41980, 'letzte_wartung': '2026-07-28', 'service': 'kein Servicevertrag'},
    {'id': 'M-04', 'name': 'Rundschleifmaschine', 'typ': 'Brenner RS 600', 'baujahr': 2019, 'standort': 'Halle 2, Platz 2', 'status': 'in Betrieb', 'betriebsstunden': 22310, 'letzte_wartung': '2026-09-15', 'service': 'kein Servicevertrag'},
    {'id': 'M-05', 'name': 'Roboter-Schweißzelle', 'typ': 'Zelle mit Varion 500', 'baujahr': 2020, 'standort': 'Halle 2, Platz 6', 'status': 'in Betrieb', 'betriebsstunden': 19870, 'letzte_wartung': '2026-09-09', 'service': 'SV-2045'},
    {'id': 'M-06', 'name': 'Montagelinie A', 'typ': 'Eigenbau, Umbau 2023', 'baujahr': 2017, 'standort': 'Halle 2, Bereich M', 'status': 'in Betrieb', 'betriebsstunden': 30560, 'letzte_wartung': '2026-08-30', 'service': 'kein Servicevertrag'},
    {'id': 'M-07', 'name': 'Prüfstand', 'typ': 'Druckprüfeinheit mit Messrechner', 'baujahr': 2022, 'standort': 'Halle 2, Bereich P', 'status': 'in Betrieb', 'betriebsstunden': 8740, 'letzte_wartung': '2026-06-20', 'service': 'Kalibrierung über QS'},
    {'id': 'M-08', 'name': 'Teilewaschanlage', 'typ': 'Durchlaufwaschanlage', 'baujahr': 2016, 'standort': 'Halle 1, Platz 12', 'status': 'in Betrieb', 'betriebsstunden': 33900, 'letzte_wartung': '2026-09-20', 'service': 'kein Servicevertrag'},
]

WARTUNGSHISTORIE = [  # (Maschine, Datum, Tätigkeit, durchgeführt von)
    ('M-01', '2026-08-12', 'Hydrauliköl gewechselt, Filter getauscht', 'Instandhaltung'),
    ('M-01', '2026-06-03', 'Abstreifer Führungsbahn X erneuert (KFZ-8810)', 'Instandhaltung'),
    ('M-01', '2026-03-18', 'Spindelrundlauf gemessen: 3 µm, in Ordnung', 'Instandhaltung'),
    ('M-02', '2026-09-02', 'Greifer des Werkzeugwechslers gereinigt, Federn getauscht (KFZ-9120)', 'Instandhaltung'),
    ('M-02', '2026-07-01', 'Messtaster kalibriert', 'Instandhaltung'),
    ('M-02', '2026-04-14', 'Kugelgewindetriebe geprüft, kein Spiel', 'Instandhaltung'),
    ('M-03', '2026-07-28', 'Spindellager nachgeschmiert (NW-LX2, 8 g je Lager)', 'Instandhaltung'),
    ('M-03', '2026-07-28', 'Spindelrundlauf gemessen: 7 µm, nahe am Grenzwert 8 µm', 'Instandhaltung'),
    ('M-03', '2026-06-30', 'Spannfutter zerlegt, gereinigt, Backen gefettet', 'Instandhaltung'),
    ('M-03', '2026-05-19', 'Spindellager nachgeschmiert (NW-LX2, 8 g je Lager)', 'Instandhaltung'),
    ('M-03', '2026-02-10', 'Hydrauliköl gewechselt, Rücklauffilter getauscht (HTD-3310)', 'Instandhaltung'),
    ('M-04', '2026-09-15', 'Neue Schleifscheibe montiert, Klangprobe in Ordnung, ausgewuchtet', 'Instandhaltung'),
    ('M-04', '2026-08-11', 'Wuchteinrichtung geprüft', 'Instandhaltung'),
    ('M-04', '2026-06-23', 'Schleifspindel auf Lagergeräusche geprüft, unauffällig', 'Instandhaltung'),
    ('M-05', '2026-09-09', 'Hydraulik der Spannvorrichtungen geprüft, kleine Leckage an Zylinder 2 abgedichtet', 'Instandhaltung'),
    ('M-05', '2026-08-05', 'Schweißstromquelle getauscht nach Ausfall', 'Servicepartner SV-2045'),
    ('M-05', '2026-05-12', 'Kühlflüssigkeit des Brenners ersetzt', 'Instandhaltung'),
    ('M-06', '2026-08-30', 'Kettenspannung Fördersystem nachgestellt', 'Instandhaltung'),
    ('M-06', '2026-07-14', 'Datensicherung der SPS nach Programmänderung', 'Instandhaltung'),
    ('M-06', '2026-06-02', 'Zentralschmierung aufgefüllt (NW-K100)', 'Instandhaltung'),
    ('M-07', '2026-06-20', 'Kalibrierung der Drucksensoren durch Dienstleister', 'Qualitätssicherung'),
    ('M-07', '2025-10-02', 'Elektrische Prüfung nach DGUV Vorschrift 3', 'Elektrofachkraft'),
    ('M-08', '2026-09-20', 'Waschbad gewechselt, Tank gereinigt', 'Instandhaltung'),
    ('M-08', '2026-08-24', 'Waschbad gewechselt, Tank gereinigt', 'Instandhaltung'),
    ('M-08', '2026-04-07', 'Heizstab entkalkt', 'Instandhaltung'),
]

TICKETS = [
    {'ticket': 'S-001', 'datum': '2026-10-05', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 45, 'status': 'geschlossen', 'beschreibung': 'Spindel läuft unrund, Lager nachgestellt'},
    {'ticket': 'S-002', 'datum': '2026-10-05', 'schicht': 'Spät', 'maschine': 'M-01', 'kategorie': 'elektrisch', 'prioritaet': 'hoch', 'stillstand_min': 20, 'status': 'geschlossen', 'beschreibung': 'Not-Aus durch defekten Türschalter ausgelöst'},
    {'ticket': 'S-003', 'datum': '2026-10-05', 'schicht': 'Nacht', 'maschine': 'M-06', 'kategorie': 'Material', 'prioritaet': 'mittel', 'stillstand_min': 35, 'status': 'geschlossen', 'beschreibung': 'Fehlende Gehäuseteile an Station 3'},
    {'ticket': 'S-004', 'datum': '2026-10-06', 'schicht': 'Früh', 'maschine': 'M-05', 'kategorie': 'elektrisch', 'prioritaet': 'hoch', 'stillstand_min': 240, 'status': 'geschlossen', 'beschreibung': 'Schweißstromquelle ausgefallen, Tausch durch Instandhaltung'},
    {'ticket': 'S-005', 'datum': '2026-10-06', 'schicht': 'Spät', 'maschine': 'M-03', 'kategorie': 'Software', 'prioritaet': 'mittel', 'stillstand_min': 60, 'status': 'geschlossen', 'beschreibung': 'CNC-Programm nach Update nicht ladbar'},
    {'ticket': 'S-006', 'datum': '2026-10-06', 'schicht': 'Spät', 'maschine': 'M-02', 'kategorie': 'mechanisch', 'prioritaet': 'niedrig', 'stillstand_min': 25, 'status': 'geschlossen', 'beschreibung': 'Werkzeugwechsler klemmt'},
    {'ticket': 'S-007', 'datum': '2026-10-07', 'schicht': 'Früh', 'maschine': 'M-04', 'kategorie': 'Bedienung', 'prioritaet': 'niedrig', 'stillstand_min': 15, 'status': 'geschlossen', 'beschreibung': 'Falsches Schleifprogramm gewählt'},
    {'ticket': 'S-008', 'datum': '2026-10-07', 'schicht': 'Nacht', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 35, 'status': 'geschlossen', 'beschreibung': 'Spänestau im Förderer'},
    {'ticket': 'S-009', 'datum': '2026-10-07', 'schicht': 'Nacht', 'maschine': 'M-07', 'kategorie': 'Software', 'prioritaet': 'mittel', 'stillstand_min': 50, 'status': 'geschlossen', 'beschreibung': 'Prüfsoftware hängt beim Messlauf'},
    {'ticket': 'S-010', 'datum': '2026-10-08', 'schicht': 'Früh', 'maschine': 'M-06', 'kategorie': 'Material', 'prioritaet': 'mittel', 'stillstand_min': 40, 'status': 'geschlossen', 'beschreibung': 'Schraubenzuführung leer'},
    {'ticket': 'S-011', 'datum': '2026-10-08', 'schicht': 'Spät', 'maschine': 'M-01', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 70, 'status': 'geschlossen', 'beschreibung': 'Kühlmittelpumpe undicht'},
    {'ticket': 'S-012', 'datum': '2026-10-08', 'schicht': 'Spät', 'maschine': 'M-08', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 30, 'status': 'geschlossen', 'beschreibung': 'Heizregister der Waschanlage ohne Funktion'},
    {'ticket': 'S-013', 'datum': '2026-10-09', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 90, 'status': 'geschlossen', 'beschreibung': 'Spindel erneut unrund, Lagerwechsel geplant'},
    {'ticket': 'S-014', 'datum': '2026-10-09', 'schicht': 'Nacht', 'maschine': 'M-05', 'kategorie': 'Bedienung', 'prioritaet': 'mittel', 'stillstand_min': 30, 'status': 'geschlossen', 'beschreibung': 'Schweißdraht falsch eingefädelt'},
    {'ticket': 'S-015', 'datum': '2026-10-12', 'schicht': 'Früh', 'maschine': 'M-02', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 55, 'status': 'geschlossen', 'beschreibung': 'Achsantrieb Y meldet Überstrom'},
    {'ticket': 'S-016', 'datum': '2026-10-12', 'schicht': 'Spät', 'maschine': 'M-06', 'kategorie': 'Bedienung', 'prioritaet': 'hoch', 'stillstand_min': 10, 'status': 'geschlossen', 'beschreibung': 'Lichtvorhang beim Teilewechsel unterbrochen'},
    {'ticket': 'S-017', 'datum': '2026-10-13', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 40, 'status': 'geschlossen', 'beschreibung': 'Sensor am Spannfutter liefert kein Signal'},
    {'ticket': 'S-018', 'datum': '2026-10-13', 'schicht': 'Nacht', 'maschine': 'M-04', 'kategorie': 'mechanisch', 'prioritaet': 'hoch', 'stillstand_min': 65, 'status': 'geschlossen', 'beschreibung': 'Schleifscheibe gebrochen, Austausch'},
    {'ticket': 'S-019', 'datum': '2026-10-14', 'schicht': 'Früh', 'maschine': 'M-07', 'kategorie': 'Software', 'prioritaet': 'niedrig', 'stillstand_min': 20, 'status': 'geschlossen', 'beschreibung': 'Messprotokoll wird nicht gespeichert'},
    {'ticket': 'S-020', 'datum': '2026-10-14', 'schicht': 'Spät', 'maschine': 'M-01', 'kategorie': 'Material', 'prioritaet': 'niedrig', 'stillstand_min': 25, 'status': 'geschlossen', 'beschreibung': 'Rohteile mit Übermaß angeliefert'},
    {'ticket': 'S-021', 'datum': '2026-10-15', 'schicht': 'Früh', 'maschine': 'M-03', 'kategorie': 'mechanisch', 'prioritaet': 'mittel', 'stillstand_min': 55, 'status': 'offen', 'beschreibung': 'Spindelgeräusch, Lagerwechsel fürs Wochenende eingeplant'},
    {'ticket': 'S-022', 'datum': '2026-10-15', 'schicht': 'Spät', 'maschine': 'M-08', 'kategorie': 'mechanisch', 'prioritaet': 'hoch', 'stillstand_min': 45, 'status': 'geschlossen', 'beschreibung': 'Förderband der Waschanlage gerissen'},
    {'ticket': 'S-023', 'datum': '2026-10-16', 'schicht': 'Nacht', 'maschine': 'M-02', 'kategorie': 'Software', 'prioritaet': 'mittel', 'stillstand_min': 35, 'status': 'geschlossen', 'beschreibung': 'Werkzeugdaten nach Neustart verloren'},
    {'ticket': 'S-024', 'datum': '2026-10-16', 'schicht': 'Spät', 'maschine': 'M-06', 'kategorie': 'elektrisch', 'prioritaet': 'mittel', 'stillstand_min': 75, 'status': 'offen', 'beschreibung': 'Schrauber 2 liefert falsches Drehmoment'},
]

TICKET_LOG = []   # hier landen neu angelegte Tickets (simuliert)
print(len(MASCHINEN), 'Maschinen,', len(WARTUNGSHISTORIE), 'Wartungseinträge,', len(TICKETS), 'Tickets')

In [ ]:
HANDBUCH = '''# Wartungshandbuch Nordwerk GmbH, Werk Karlsruhe (Auszug)

Dokument WH-2026-03 · Stand: September 2026 · Herausgeber: Abteilung Instandhaltung · Dieses Dokument ist fiktiv und dient ausschließlich Schulungszwecken.

## 1 Allgemeines

### 1.1 Geltungsbereich und Aufbau

Dieses Handbuch gilt für alle Fertigungs- und Prüfanlagen in den Hallen 1 und 2 des Werks Karlsruhe. Es richtet sich an Maschinenbedienerinnen und -bediener, Schichtleitungen und das Instandhaltungsteam. Kapitel 1 regelt Zuständigkeiten, Sicherheit und Dokumentation. Kapitel 2 beschreibt Betriebsstoffe, die an mehreren Maschinen verwendet werden. Kapitel 3 enthält für jede Maschine eine Kurzbeschreibung, die Wartungsintervalle, typische Störungen und wichtige Ersatzteile. Kapitel 4 regelt das Ersatzteillager, Kapitel 5 enthält Fehlercodes und Kontakte.

Die Herstellerdokumentation der einzelnen Maschinen bleibt verbindlich. Bei Widersprüchen gilt die Herstellerangabe, und die Instandhaltung ist zu informieren, damit dieses Handbuch angepasst werden kann. Änderungen am Handbuch gibt die Leitung Instandhaltung frei. Die jeweils gültige Fassung liegt im Intranet. Ausgedruckte Exemplare an den Maschinen sind nach jeder Änderung auszutauschen.

### 1.2 Zuständigkeiten und Eskalation

Für die tägliche Pflege und die Sichtprüfungen ist die Bedienerin oder der Bediener der Maschine zuständig. Wartungsarbeiten mit Werkzeug, das Öffnen von Schaltschränken und alle Arbeiten an Hydraulik, Druckluftverteilung und Elektrik führt ausschließlich das Instandhaltungsteam durch.

Bei einer Störung meldet die Bedienerin oder der Bediener zunächst an die Schichtleitung. Die Schichtleitung entscheidet über die Priorität und informiert die Instandhaltung. Während der Kernzeit von 6:00 bis 22:00 Uhr ist die Instandhaltung über die Leitstelle erreichbar. Außerhalb der Kernzeit, an Wochenenden und an Feiertagen ist bei Störungen mit hoher Priorität die Rufbereitschaft der Instandhaltung unter der internen Nummer 4711 anzurufen. Bei Störungen mit mittlerer oder niedriger Priorität genügt außerhalb der Kernzeit ein Ticket, das am nächsten Arbeitstag bearbeitet wird.

Kann die Instandhaltung eine Störung nicht innerhalb von vier Stunden beheben, informiert sie die Produktionsleitung und entscheidet gemeinsam mit ihr, ob ein externer Servicepartner beauftragt wird. Die Kontakte der Servicepartner stehen in Kapitel 5.2.

### 1.3 Arbeitssicherheit

Vor jeder Wartung, Reinigung oder Störungsbehebung im Gefahrenbereich einer Maschine ist die Maschine freizuschalten. Das Vorgehen folgt immer denselben fünf Schritten: Maschine stillsetzen, Hauptschalter ausschalten, Hauptschalter mit persönlichem Vorhängeschloss gegen Wiedereinschalten sichern, Spannungsfreiheit und Druckfreiheit prüfen, Arbeitsbereich kennzeichnen. Jede Person, die im Gefahrenbereich arbeitet, hängt ihr eigenes Schloss an. Die Maschine darf erst wieder eingeschaltet werden, wenn alle Schlösser entfernt sind.

Arbeiten an Hydraulikanlagen sind nur im drucklosen Zustand erlaubt. Druckspeicher müssen vorher entladen werden, der Manometerwert muss null zeigen. Arbeiten in der Roboter-Schweißzelle erfolgen grundsätzlich zu zweit, eine Person bleibt außerhalb der Zelle und hat Zugriff auf den Not-Aus. Beim Umgang mit Kühlschmierstoff sind Schutzhandschuhe aus Nitril und eine Schutzbrille zu tragen. Hautschutzplan und Betriebsanweisung hängen an jeder Maschine mit Kühlschmierstoff aus.

Defekte Schutzeinrichtungen wie Türschalter, Lichtgitter oder Not-Aus-Taster dürfen niemals überbrückt werden. Ist eine Schutzeinrichtung defekt, bleibt die Maschine außer Betrieb, bis die Instandhaltung sie instand gesetzt und geprüft hat.

### 1.4 Dokumentation von Störungen und Wartungen

Jede Störung wird als Ticket im Instandhaltungssystem erfasst. Störungen mit hoher Priorität sind innerhalb von 30 Minuten nach Feststellung als Ticket anzulegen, alle anderen bis zum Ende der Schicht. Das Ticket enthält Maschine, Zeitpunkt, Beschreibung, vermutete Ursache, Priorität und Stillstandszeit.

Für die Priorität gelten im ganzen Werk dieselben Definitionen. Hoch bedeutet: Die Maschine steht oder es besteht ein Sicherheitsrisiko. Mittel bedeutet: Die Maschine läuft eingeschränkt, Qualität oder Tempo leiden. Niedrig bedeutet: kein Einfluss auf die laufende Produktion.

Durchgeführte Wartungen werden mit Datum, Tätigkeit, verbrauchten Ersatzteilen und Namen im Wartungsplan der Maschine eingetragen. Bei Abweichungen vom Sollzustand, etwa ungewöhnlichem Verschleiß, ist zusätzlich ein Hinweis-Ticket anzulegen, damit die Intervalle überprüft werden können.

## 2 Betriebsstoffe

### 2.1 Schmierstoffe und Öle

Im Werk werden nur die folgenden freigegebenen Betriebsstoffe verwendet. Andere Produkte dürfen nur nach Freigabe durch die Instandhaltung eingesetzt werden, weil falsche Schmierstoffe Dichtungen angreifen und Lager schädigen können.

| Anwendung | Bezeichnung | Gebinde | Lagerort |
|---|---|---|---|
| Hydraulikaggregate aller Maschinen | Hydrauliköl HLP 46 | 20-Liter-Kanister | Gefahrstofflager, Regal G-2 |
| Spindellager der Drehmaschine | Wälzlagerfett NW-LX2 | 400-Gramm-Kartusche | Ersatzteillager, Regal C-04 |
| Führungsbahnen der Fräszentren | Gleitbahnöl CGLP 68 | 20-Liter-Kanister | Gefahrstofflager, Regal G-3 |
| Zentralschmierung der Montagelinie | Kettenöl NW-K100 | 5-Liter-Kanister | Gefahrstofflager, Regal G-3 |
| Kühlschmierstoff aller Zerspanungsmaschinen | Emulsion NW-Cut 300 (Konzentrat) | 200-Liter-Fass | Gefahrstofflager, Bereich G-1 |

Leere Gebinde und Altöl werden in den gekennzeichneten Sammelbehältern im Gefahrstofflager entsorgt. Ölgetränkte Putzlappen gehören in die roten Sicherheitsbehälter an den Maschinen.

### 2.2 Pflege des Kühlschmierstoffs

Die Fräszentren, die Drehmaschine und die Rundschleifmaschine arbeiten mit der wassergemischten Emulsion NW-Cut 300. Ein gut gepflegter Kühlschmierstoff verlängert die Standzeit der Werkzeuge, verbessert die Oberflächengüte und schützt die Haut der Beschäftigten.

Die Konzentration soll zwischen 6 bis 8 Prozent liegen. Sie wird einmal pro Woche mit dem Refraktometer gemessen, der Messwert wird mit dem Faktor 1,1 multipliziert (Refraktometerfaktor des Konzentrats). Der pH-Wert soll zwischen 8,5 und 9,3 liegen, der Nitritgehalt unter 20 mg/l. Alle drei Werte werden im Kühlschmierstoff-Protokoll an der Maschine eingetragen.

Ist die Konzentration zu niedrig, wird Konzentrat über das Mischgerät nachdosiert, nie unverdünnt in den Tank gegeben. Ist sie zu hoch, wird mit Wasser nachgefüllt. Schäumt der Kühlschmierstoff stark, ist zuerst der Füllstand zu prüfen, denn zu wenig Flüssigkeit im Tank führt zu Luftansaugung an der Pumpe. Hilft das nicht, wird nach Rücksprache mit der Instandhaltung Entschäumer NW-AF 2 in einer Dosierung von 0,05 Prozent zugegeben. Riecht der Kühlschmierstoff faulig oder liegt der pH-Wert unter 8,0, ist er biologisch belastet und muss gewechselt werden. Unabhängig davon wird der Kühlschmierstoff spätestens alle sechs Monate vollständig gewechselt, einschließlich Reinigung des Tanks.

### 2.3 Druckluft

Die zentrale Druckluftversorgung liefert einen Netzdruck von 6 bar. An jeder Maschine sitzt eine Wartungseinheit aus Filter, Druckregler und Wasserabscheider. Der Wasserabscheider wird täglich zu Schichtbeginn entleert, sofern er keinen automatischen Ablass hat. Das Filterelement wird halbjährlich gewechselt oder früher, wenn die Differenzdruckanzeige rot zeigt.

Fällt der Druck an einer Maschine unter 5,5 bar, schaltet die Maschinensteuerung in den meisten Fällen mit einer Warnung ab. Druckschwankungen haben häufig eine von drei Ursachen: ein verschmutztes Filterelement, eine Leckage an Schläuchen oder Kupplungen oder ein zu hoher gleichzeitiger Verbrauch in der Halle. Leckagen lassen sich mit Lecksuchspray finden. Undichte Kupplungen werden getauscht, nicht abgedichtet.

## 3 Maschinen

### 3.1 M-01 CNC-Fräszentrum 1

#### Kurzbeschreibung

Das CNC-Fräszentrum 1 ist ein vertikales Bearbeitungszentrum vom Typ Kessler KFZ 800, Baujahr 2018, Standort Halle 1, Platz 4. Es bearbeitet vor allem Gehäuseteile aus Aluminium. Die Arbeitsspindel erreicht 12.000 Umdrehungen pro Minute. Das Werkzeugmagazin hat 24 Plätze. Verfahrwege: X 800 mm, Y 500 mm, Z 500 mm.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Späne aus dem Arbeitsraum entfernen, Sichtfenster reinigen | täglich | Bedienung |
| Füllstand Gleitbahnöl prüfen | täglich | Bedienung |
| Kühlschmierstoff messen (Kapitel 2.2) | wöchentlich | Bedienung |
| Abstreifer der Führungsbahnen prüfen | monatlich | Instandhaltung |
| Spindelrundlauf mit Prüfdorn messen, Grenzwert 5 µm | alle 1.000 Betriebsstunden | Instandhaltung |
| Hydrauliköl wechseln, Filter tauschen | jährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Achsantrieb meldet Überstrom | Schwergängige Führung, Späne unter dem Abstreifer, Motor defekt | Maschine freischalten, Abstreifer reinigen; tritt der Fehler erneut auf, Instandhaltung rufen |
| Kühlmittel tropft aus Leitung | Schlauchschelle locker, Schlauch porös | Auffangwanne unterstellen, Ticket mit Priorität niedrig, Schlauch beim nächsten Stillstand tauschen |
| CNC-Programm lässt sich nicht laden | Programm fehlerhaft übertragen, Speicher voll | Programm erneut aus der Programmverwaltung laden, alte Programme archivieren |

#### Wichtige Ersatzteile

| Teil | Teilenummer | Lagerort |
|---|---|---|
| Abstreifer Führungsbahn X | KFZ-8810 | Regal A-03 |
| Kühlmittelschlauch 12 mm, Meterware | NW-SL12 | Regal A-07 |

### 3.2 M-02 CNC-Fräszentrum 2

#### Kurzbeschreibung

Das CNC-Fräszentrum 2 ist baugleich zum Fräszentrum 1, jedoch vom neueren Typ Kessler KFZ 800 plus, Baujahr 2021, Standort Halle 1, Platz 5. Unterschiede: Das Werkzeugmagazin hat 40 Plätze statt 24, und die Maschine besitzt einen Messtaster zur automatischen Werkstückvermessung. Es fertigt vorwiegend Teile aus Stahl.

#### Wartungsintervalle

Es gelten dieselben Intervalle wie bei M-01. Zusätzlich gilt:

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Greifer des Werkzeugwechslers reinigen und auf Verschleiß prüfen | monatlich | Instandhaltung |
| Messtaster kalibrieren | vierteljährlich | Instandhaltung |
| Kugelgewindetriebe auf Spiel prüfen | alle 2.000 Betriebsstunden | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Werkzeugwechsler hakt, Wechsel dauert deutlich länger | Verschmutzte Greiferfinger, verschlissene Federn | Greifer reinigen; bei Verschleiß Federn tauschen (Teilenummer KFZ-9120) |
| Quietschen an einer Achse | Schmierung der Führung unzureichend | Zentralschmierung manuell auslösen, Gleitbahnöl-Füllstand prüfen; Geräusch weiter beobachten |
| Messtaster liefert stark streuende Werte | Tastkugel verschmutzt, Kalibrierung veraltet | Tastkugel reinigen, Messtaster neu kalibrieren |

### 3.3 M-03 CNC-Drehmaschine

#### Kurzbeschreibung

Die CNC-Drehmaschine ist eine Schrägbettmaschine vom Typ Hartwig TD 400, Baujahr 2015, Standort Halle 1, Platz 9. Sie fertigt Wellen und Bolzen bis 400 mm Länge. Das Hydraulikaggregat versorgt das Spannfutter und den Reitstock. Wegen ihres Alters ist sie die störanfälligste Maschine in Halle 1.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Späne entfernen, Spannfutter abblasen | täglich | Bedienung |
| Ölstand im Hydraulikaggregat prüfen (Hydrauliköl HLP 46) | täglich | Bedienung |
| Spindellager mit Wälzlagerfett NW-LX2 nachschmieren, Menge 8 Gramm je Lager | alle 500 Betriebsstunden | Instandhaltung |
| Spannfutter zerlegen, reinigen, Backen fetten | alle 250 Betriebsstunden | Instandhaltung |
| Spindelrundlauf messen, Grenzwert 8 µm | alle 1.000 Betriebsstunden | Instandhaltung |
| Hydrauliköl wechseln, Rücklauffilter tauschen | jährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Spindel vibriert, Teile außerhalb der Toleranz | Spindellager verschlissen oder unzureichend geschmiert | Drehzahl reduzieren, Instandhaltung informieren; Rundlauf messen, Lager nachschmieren oder tauschen |
| Maschine schaltet mit Meldung „Hydraulikdruck“ ab | Ölstand unter Minimum, Leckage | Ölstand prüfen; nur mit HLP 46 auffüllen; Leckage suchen, Ticket mit Priorität hoch |
| Spannfutter spannt ungleichmäßig | Späne in der Backenführung | Spannfutter reinigen; bei Wiederholung zerlegen |

#### Wichtige Ersatzteile

| Teil | Teilenummer | Lagerort |
|---|---|---|
| Ersatz-Spannfutter dreibackig, 250 mm | HTD-2250 | Regal C-12 |
| Spindellagersatz | HTD-1100 | Regal C-11 |
| Rücklauffilter Hydraulik | HTD-3310 | Regal C-04 |

### 3.4 M-04 Rundschleifmaschine

#### Kurzbeschreibung

Die Rundschleifmaschine vom Typ Brenner RS 600, Baujahr 2019, Standort Halle 2, Platz 2, schleift gehärtete Wellen auf das Endmaß. Die Schleifscheibe hat einen Durchmesser von 500 mm. Der Kühlschmierstoff wird über ein Papierbandfilter gereinigt.

#### Sicherheitshinweise zur Schleifscheibe

Schleifscheiben sind bruchgefährdet. Eine neue Schleifscheibe wird vor der Montage durch eine Klangprobe geprüft: Die Scheibe wird frei hängend mit einem Holzhammer leicht angeschlagen. Ein heller, klarer Klang zeigt eine unbeschädigte Scheibe, ein dumpfer oder scheppernder Klang deutet auf Risse hin. Scheiben mit Rissen oder unklarer Klangprobe dürfen nicht montiert werden. Die zulässige Umfangsgeschwindigkeit der eingesetzten Scheiben beträgt höchstens 50 m/s. Nach jeder Montage wird die Scheibe ausgewuchtet und abgerichtet, anschließend läuft sie mindestens fünf Minuten mit Betriebsdrehzahl, während sich niemand in der Drehebene aufhält.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Schleifscheibe abrichten | je nach Bedarf, spätestens alle 50 Werkstücke | Bedienung |
| Papierbandfilter prüfen, Rolle wechseln | wenn die Füllstandsanzeige gelb zeigt | Bedienung |
| Kühlschmierstoff messen (Kapitel 2.2) | wöchentlich | Bedienung |
| Wuchteinrichtung prüfen | monatlich | Instandhaltung |
| Schleifspindel auf Lagergeräusche prüfen | vierteljährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Risse an der Schleifscheibe | Schlag, Überlastung, falsche Lagerung | Maschine sofort stoppen und freischalten, Scheibe tauschen, Ticket mit Priorität hoch |
| Rattermarken auf dem Werkstück | Scheibe unwuchtig oder stumpf | Scheibe auswuchten und abrichten |
| Englische Bedienoberfläche nach Update | Spracheinstellung zurückgesetzt | Im Menü Einstellungen, Sprache auf Deutsch stellen; kein Einfluss auf die Produktion |

### 3.5 M-05 Roboter-Schweißzelle

#### Kurzbeschreibung

Die Roboter-Schweißzelle besteht aus einem Sechsachs-Roboter, der Schweißstromquelle Varion 500, einem Drehtisch mit zwei hydraulischen Spannvorrichtungen und einem Lichtgitter an der Beladeseite. Baujahr 2020, Standort Halle 2, Platz 6. Geschweißt wird im MAG-Verfahren mit Mischgas.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Brenner reinigen, Gasdüse und Kontaktrohr prüfen | alle vier Stunden Schweißbetrieb | Bedienung |
| Lichtgitter mit Prüfstab testen | täglich zu Schichtbeginn | Bedienung |
| Schutzgasflasche wechseln | wenn der Flaschendruck unter 10 bar fällt | Bedienung |
| Kühlflüssigkeit des Brenners prüfen | wöchentlich | Bedienung |
| Hydraulik der Spannvorrichtungen auf Leckagen prüfen | monatlich | Instandhaltung |
| Roboter-Getriebeöl wechseln | alle 20.000 Betriebsstunden | Servicepartner |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Schweißstromquelle zeigt Fehlercode | siehe Kapitel 5.1 | Fehlercode ablesen und nach Tabelle vorgehen |
| Lichtgitter löst aus, obwohl sich niemand im Schutzfeld befindet | Verschmutzte Linsen, Schweißspritzer auf Sender oder Empfänger, verstellte Ausrichtung | Linsen mit weichem Tuch reinigen, Ausrichtung prüfen; Lichtgitter niemals überbrücken; bei Wiederholung Instandhaltung rufen |
| Spannvorrichtung öffnet sich während des Schweißens | Hydraulikdruck fällt ab, Leckage am Spannzylinder | Zelle sofort stillsetzen, Ticket mit Priorität hoch, Verletzungsgefahr |
| Bildschirm am Bedienpult flackert | Wackelkontakt im Displaykabel | Ticket mit Priorität niedrig, Bedienung bleibt möglich |

### 3.6 M-06 Montagelinie A

#### Kurzbeschreibung

Die Montagelinie A verbindet sechs Montagestationen über ein Werkstückträger-Fördersystem. Gesteuert wird sie von einer zentralen SPS, die über das Werksnetz mit dem Leitrechner der Produktion verbunden ist. Lichtschranken an der Zuführung erkennen die Werkstückträger. Baujahr 2017, Umbau 2023, Standort Halle 2, Bereich M.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Lichtschranken an Zuführung und Stationen reinigen | täglich | Bedienung |
| Not-Aus-Taster aller Stationen auf Funktion prüfen | wöchentlich | Schichtleitung |
| Kettenspannung des Fördersystems prüfen | monatlich | Instandhaltung |
| Zentralschmierung auffüllen (Kettenöl NW-K100) | wenn die Anzeige rot zeigt | Instandhaltung |
| Datensicherung der SPS | nach jeder Programmänderung, mindestens vierteljährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Not-Aus lässt sich nicht zurücksetzen | Taster mechanisch verklemmt, Fehler im Sicherheitskreis | Alle Not-Aus-Taster der Linie prüfen; bleibt der Fehler, Instandhaltung rufen; Linie steht, Priorität hoch |
| SPS verliert die Verbindung zum Leitrechner | Netzwerkkabel, Switch, IP-Konflikt | Linkanzeige am Switch prüfen; Neustart der SPS nur durch die Instandhaltung |
| Kurze Unterbrechungen durch Lichtschranke | Staub auf der Optik | Lichtschranke reinigen, bei Wiederholung Ausrichtung prüfen |

### 3.7 M-07 Prüfstand

#### Kurzbeschreibung

Der Prüfstand prüft fertig montierte Baugruppen auf Dichtheit und Funktion. Er besteht aus einer Druckprüfeinheit mit Druckluft, einem Messrechner mit Prüfsoftware und einem Etikettendrucker. Baujahr 2022, Standort Halle 2, Bereich P. Die Messergebnisse werden automatisch auf dem Qualitätsserver gespeichert.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Prüfadapter und Dichtungen auf Beschädigung prüfen | täglich | Bedienung |
| Leckprüfung mit Referenzteil | zu jedem Schichtbeginn | Bedienung |
| Kalibrierung der Drucksensoren | jährlich durch einen akkreditierten Dienstleister | Qualitätssicherung |
| Elektrische Prüfung nach DGUV Vorschrift 3 | jährlich | Elektrofachkraft |
| Datensicherung des Messrechners prüfen | monatlich | IT |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Messprotokolle werden nicht gespeichert | Verbindung zum Qualitätsserver gestört, Speicher voll | Werte von Hand im Ersatzprotokoll notieren, IT informieren, Priorität mittel |
| Messungen streuen, Referenzteil fällt durch | Druckluft schwankt, Dichtung am Adapter verschlissen | Netzdruck an der Wartungseinheit prüfen (Kapitel 2.3), Dichtung tauschen |
| Prüfplakette abgelaufen | Prüftermin verpasst | Elektrofachkraft informieren, Prüfung nachholen |

### 3.8 M-08 Teilewaschanlage

#### Kurzbeschreibung

Die Teilewaschanlage reinigt Teile nach der Zerspanung von Spänen und Kühlschmierstoff. Sie arbeitet mit wässriger, alkalischer Waschlauge bei 60 °C, die über einen elektrischen Heizstab erwärmt wird. Ein Förderband transportiert die Teilekörbe durch Wasch-, Spül- und Trockenzone. Baujahr 2016, Standort Halle 1, Platz 12.

#### Wartungsintervalle

| Tätigkeit | Intervall | Zuständig |
|---|---|---|
| Siebe in der Waschzone reinigen | täglich | Bedienung |
| Temperatur und Füllstand der Waschlauge prüfen | täglich | Bedienung |
| Ölabscheider leeren | wöchentlich | Bedienung |
| Waschbad komplett wechseln, Tank reinigen | alle vier Wochen | Instandhaltung |
| Förderband auf Spannung und Verschleiß prüfen | monatlich | Instandhaltung |
| Heizstab entkalken | halbjährlich | Instandhaltung |

#### Typische Störungen

| Symptom | Mögliche Ursache | Maßnahme |
|---|---|---|
| Teile bleiben fettig | Waschlauge zu kalt oder verbraucht | Temperatur prüfen; liegt sie deutlich unter 60 °C, Heizstab und Sicherung prüfen lassen; Bad wechseln, wenn es älter als vier Wochen ist |
| Förderband steht | Band gerissen oder Antrieb blockiert | Anlage freischalten, Ursache prüfen, Priorität hoch |
| Waschzyklen zu kurz nach Stromausfall | Zeitsteuerung auf Werkseinstellung zurückgesetzt | Parameter aus der Parameterliste im Schaltschrank neu eingeben |

## 4 Ersatzteillager und Bestellung

Das Ersatzteillager befindet sich in Halle 2 neben der Werkstatt der Instandhaltung. Die Regale sind nach Maschinen gruppiert: Regalreihe A für die Fräszentren, B für die Schleifmaschine, C für die Drehmaschine, D für die Schweißzelle, E für Montagelinie und Prüfstand, F für die Waschanlage. Gefahrstoffe lagern getrennt im Gefahrstofflager.

Für kritische Teile gelten Mindestbestände. Wird ein Teil entnommen, trägt die entnehmende Person die Entnahme in die Lagerliste ein. Fällt der Bestand unter den Mindestbestand, bestellt die Instandhaltung über den Einkauf nach. Bestellungen bis 500 Euro gibt die Leitung Instandhaltung frei, darüber die Werkleitung. Für Eilbestellungen bei Stillstand mit hoher Priorität darf die Rufbereitschaft auch außerhalb der Kernzeit direkt beim Servicepartner bestellen und dokumentiert dies am nächsten Arbeitstag.

| Teil | Mindestbestand | Übliche Lieferzeit |
|---|---|---|
| Schleifscheibe 500 mm für M-04 | 2 Stück | 3 Wochen |
| Spindellagersatz HTD-1100 für M-03 | 1 Satz | 6 Wochen |
| Kontaktrohre für den Schweißbrenner | 50 Stück | 1 Woche |
| Lichtschranke Zuführung M-06 | 1 Stück | 2 Wochen |

## 5 Anhang

### 5.1 Fehlercodes der Schweißstromquelle Varion 500

| Code | Bedeutung | Maßnahme |
|---|---|---|
| E10 | Netzspannung außerhalb des zulässigen Bereichs | Netzversorgung prüfen lassen |
| E21 | Übertemperatur im Kühlkreislauf des Brenners | Kühlflüssigkeit prüfen und auffüllen, Kühler auf Verschmutzung prüfen; Anlage abkühlen lassen |
| E33 | Drahtvorschub blockiert | Drahtführung und Vorschubrollen prüfen, Drahtspule auf Verwicklungen prüfen |
| E41 | Kein Gasdurchfluss | Flaschenventil und Druckminderer prüfen, Flasche bei unter 10 bar wechseln |
| E45 | Kommunikation zur Robotersteuerung unterbrochen | Steckverbindung des Buskabels prüfen, Instandhaltung rufen |

### 5.2 Kontakte

| Bereich | Kontakt |
|---|---|
| Leitstelle Instandhaltung (Kernzeit 6 bis 22 Uhr) | intern 4700 |
| Rufbereitschaft Instandhaltung (außerhalb der Kernzeit) | intern 4711 |
| IT-Service | intern 4800 |
| Servicepartner Fräszentren (Kessler Service) | Servicevertrag SV-1180, Hotline im Intranet |
| Servicepartner Schweißzelle | Servicevertrag SV-2045, Hotline im Intranet |
| Kalibrierdienstleister Prüfstand | Auftrag über die Qualitätssicherung |
'''
print(len(HANDBUCH), 'Zeichen Handbuch')

---
# Teil A · Ein Tool, ein Durchlauf von Hand

## A1 · Die Python-Funktion

Ein Tool ist zuerst eine ganz normale Python-Funktion. Wichtig ist eine **hilfreiche Fehlermeldung**: Sie geht später an das Modell zurück, und das Modell soll daraus lernen können, was falsch war.

### TODO 1 · `get_machine_status`
Sucht die Maschine in `MASCHINEN` und gebt ein Dictionary zurück mit `id`, `name`, `status`, `betriebsstunden`, `letzte_wartung` und `offene_tickets` (Anzahl der Tickets in `TICKETS` mit dieser Maschine und Status `offen`).
Bei unbekannter ID: `raise ValueError(...)` mit einer Nachricht, die die **gültigen IDs nennt**. Die Hilfsfunktion `_maschine` ist vorgegeben.

In [ ]:
GUELTIGE_IDS = [m["id"] for m in MASCHINEN]

def _maschine(machine_id):
    for m in MASCHINEN:
        if m["id"] == machine_id:
            return m
    raise ValueError(f"Unbekannte Maschinen-ID '{machine_id}'. Gültig sind: {', '.join(GUELTIGE_IDS)}.")

def get_machine_status(machine_id):
    # TODO 1
    pass

In [ ]:
def _wirft(f, *a):
    try:
        f(*a); return None
    except ValueError as e:
        return str(e)

check("TODO 1a Status M-03", lambda: get_machine_status("M-03")["status"] == "eingeschränkt" and get_machine_status("M-03")["offene_tickets"] == 1)
check("TODO 1b alle Felder", lambda: set(get_machine_status("M-01")) == {"id", "name", "status", "betriebsstunden", "letzte_wartung", "offene_tickets"})
check("TODO 1c hilfreicher Fehler bei M-12", lambda: "M-08" in (_wirft(get_machine_status, "M-12") or ""))

## A2 · Die Tool-Definition

Das Modell sieht nie euren Python-Code, sondern nur die **Definition**: Name, Beschreibung und ein JSON-Schema für die Parameter. Die Beschreibung ist selbst ein Prompt. Sie sollte sagen, **was** das Tool tut, **wann** es benutzt werden soll und **was** zurückkommt.

### TODO 2 · `TOOL_STATUS`
Ergänzt Beschreibung und Schema. Der Parameter `machine_id` ist ein String und Pflicht. Nennt in seiner Beschreibung das Format (`M-01` bis `M-08`). Setzt außerdem `"additionalProperties": False`: Dann sind keine erfundenen Zusatzparameter erlaubt. Die Schnittstelle erwartet die Definition später in einer Hülle `{"type": "function", "function": …}`, das erledigt die Hilfsfunktion `als_tools`.

In [ ]:
TOOL_STATUS = {
    "name": "get_machine_status",
    "description": "",          # TODO 2a
    "parameters": {},         # TODO 2b
}

In [ ]:
check("TODO 2a Beschreibung sagt was und wann", lambda: len(TOOL_STATUS["description"]) >= 80)
check("TODO 2b Schema mit Pflichtparameter", lambda: TOOL_STATUS["parameters"]["required"] == ["machine_id"] and TOOL_STATUS["parameters"]["properties"]["machine_id"]["type"] == "string")
check("TODO 2c keine Zusatzparameter", lambda: TOOL_STATUS["parameters"]["additionalProperties"] is False)

## A3 · Ein Durchlauf, Schritt für Schritt

### TODO 3 · Das Ergebnis zurückmelden
Ein Tool-Ergebnis ist eine eigene Nachricht mit der Rolle `"tool"`. Sie braucht die `tool_call_id` des Aufrufs und den Inhalt als **String** (Tipp: `json.dumps(..., ensure_ascii=False)`). Ein Fehlerkennzeichen gibt es in diesem Format nicht. Wir stellen bei Fehlern deshalb `"FEHLER: "` vor den Inhalt, damit das Modell es sicher erkennt.

In [ ]:
def tool_nachricht(tool_call_id, ergebnis, is_error=False):
    # TODO 3
    pass

In [ ]:
_b = tool_nachricht("call_123", {"status": "in Betrieb"})
check("TODO 3a Rolle und ID", lambda: _b["role"] == "tool" and _b["tool_call_id"] == "call_123")
check("TODO 3b Inhalt als JSON-String", lambda: isinstance(_b["content"], str) and json.loads(_b["content"]) == {"status": "in Betrieb"})
check("TODO 3c Fehlerkennzeichen", lambda: tool_nachricht("x", {"fehler": "…"}, True)["content"].startswith("FEHLER"))

Jetzt der Durchlauf. Führt die Zellen einzeln aus und lest die Ausgaben.

In [ ]:
# Schritt 1: Frage mit Tool-Definition schicken
BEREIT_A = bool(TOOL_STATUS["description"]) and tool_nachricht("x", {}) is not None and get_machine_status("M-01") is not None
if not BEREIT_A:
    print("⏳ Erst TODO 1 bis 3 lösen.")
else:
    nachrichten = [{"role": "user", "content": "Läuft die Drehmaschine M-03 gerade normal?"}]
    antwort1 = client.chat.completions.create(model=MODEL, max_tokens=500, tools=als_tools([TOOL_STATUS]), messages=nachrichten)
    msg1 = antwort1.choices[0].message
    print("finish_reason:", antwort1.choices[0].finish_reason)
    print("  Text:", msg1.content)
    for tc in msg1.tool_calls or []:
        print("  Tool-Aufruf:", tc.function.name, tc.function.arguments, " id =", tc.id)

In [ ]:
# Schritt 2: WIR führen das Tool aus und schicken das Ergebnis zurück
if BEREIT_A:
  nachrichten.append(assistent_nachricht(msg1))
  aufruf = msg1.tool_calls[0]
  ergebnis = get_machine_status(**json.loads(aufruf.function.arguments))   # Argumente kommen als JSON-String
  print("Ergebnis unserer Funktion:", ergebnis)
  nachrichten.append(tool_nachricht(aufruf.id, ergebnis))

In [ ]:
# Schritt 3: Das Modell formuliert die Antwort
if BEREIT_A:
    antwort2 = client.chat.completions.create(model=MODEL, max_tokens=500, tools=als_tools([TOOL_STATUS]), messages=nachrichten)
    print("finish_reason:", antwort2.choices[0].finish_reason)
    print(antworttext(antwort2))
    print("\nDie ganze Nachrichtenliste:")
    zeige_nachrichten(nachrichten + [assistent_nachricht(antwort2.choices[0].message)])

> **Zum Nachdenken:** Wo genau wurde `get_machine_status` ausgeführt? Was hätte passieren können, wenn ihr statt `get_machine_status` versehentlich eine Funktion aufgerufen hättet, die Daten löscht?

---
# Teil B · Mehr Tools und eine Schleife

## B1 · Zwei weitere lesende Tools (vorgegeben)

In [ ]:
def get_maintenance_history(machine_id, limit=5):
    _maschine(machine_id)                                             # wirft bei unbekannter ID
    eintraege = sorted((w for w in WARTUNGSHISTORIE if w[0] == machine_id), key=lambda w: w[1], reverse=True)
    return [{"datum": d, "taetigkeit": t, "durchgefuehrt_von": v} for _, d, t, v in eintraege[:limit]]

TOOL_HISTORIE = {
    "name": "get_maintenance_history",
    "description": ("Liefert die letzten durchgeführten Wartungen einer Maschine, neueste zuerst, mit Datum, Tätigkeit und "
                    "ausführender Stelle. Nutze dieses Tool für Fragen wie 'Wann wurde zuletzt …?' oder 'Was wurde an M-03 gemacht?'. "
                    "Für Soll-Intervalle laut Handbuch nutze search_knowledge_base."),
    "parameters": {
        "type": "object",
        "properties": {
            "machine_id": {"type": "string", "description": "Maschinen-ID im Format M-01 bis M-08."},
            "limit": {"type": "integer", "description": "Maximale Anzahl Einträge, Standard 5."},
        },
        "required": ["machine_id"],
    },
}
print(get_maintenance_history("M-03", 2))

**Das RAG aus Woche 7 als Tool.** Für einen leichten Start nutzt die Suche hier BM25 über die Struktur-Chunks. Im Projekt setzt ihr eure E5-Suche aus Woche 7 ein.

In [ ]:
def chunk_struktur(markdown):
    chunks, pfad, zeilen, titel = [], {}, [], ""
    def abschliessen():
        inhalt = "\n".join(zeilen).strip()
        if inhalt:
            chunks.append({"titel": titel, "text": inhalt})
    for zeile in markdown.splitlines():
        if zeile.startswith("#"):
            abschliessen()
            ebene = len(zeile) - len(zeile.lstrip("#"))
            pfad = {e: t for e, t in pfad.items() if e < ebene}
            pfad[ebene] = zeile.lstrip("#").strip()
            titel = " › ".join(pfad[e] for e in sorted(pfad) if e > 1) or pfad[ebene]
            zeilen = []
        else:
            zeilen.append(zeile)
    abschliessen()
    return chunks

CHUNKS = chunk_struktur(HANDBUCH)
_tok = lambda t: re.findall(r"[a-zäöüß0-9-]+", t.lower())
_bm25 = BM25Okapi([_tok(c["titel"] + " " + c["text"]) for c in CHUNKS])

def search_knowledge_base(query, k=3):
    scores = _bm25.get_scores(_tok(query))
    besten = sorted(range(len(CHUNKS)), key=lambda i: scores[i], reverse=True)[:k]
    return [{"quelle": CHUNKS[i]["titel"], "text": CHUNKS[i]["text"]} for i in besten]

print([r["quelle"] for r in search_knowledge_base("Spindellager nachschmieren")])

### TODO 4 · `TOOL_SUCHE`
Definiert das Such-Tool. Die Beschreibung entscheidet, **wann** das Modell es benutzt. Nennt die Themen des Handbuchs (Wartungsintervalle, Sicherheit, Störungsbehebung, Betriebsstoffe, Ersatzteile, Fehlercodes) und grenzt es gegen die Status-Tools ab. Parameter: `query` (String, Pflicht).

In [ ]:
TOOL_SUCHE = {
    "name": "search_knowledge_base",
    # TODO 4
}

In [ ]:
check("TODO 4a Beschreibung nennt das Handbuch", lambda: "handbuch" in TOOL_SUCHE["description"].lower())
check("TODO 4b Abgrenzung zum Status-Tool", lambda: "get_machine_status" in TOOL_SUCHE["description"] or "status" in TOOL_SUCHE["description"].lower())
check("TODO 4c Pflichtparameter query", lambda: TOOL_SUCHE["parameters"]["required"] == ["query"])

## B2 · Ein schreibendes Tool

### TODO 5 · `create_ticket`
Die Funktion prüft die Maschinen-ID (mit `_maschine`) und die Priorität (`hoch`, `mittel` oder `niedrig`, sonst `ValueError`). Dann hängt sie ein Ticket an `TICKET_LOG` an: `{"ticket": "T-1001", "maschine": ..., "beschreibung": ..., "prioritaet": ..., "angelegt": <Zeitstempel>}`, wobei die Nummer mit jedem Ticket um 1 steigt. Zurück kommt `{"ticket": <Nummer>, "status": "angelegt"}`.

Die Definition ist vorgegeben. Beachtet das `enum` für die Priorität.

In [ ]:
def create_ticket(machine_id, description, priority):
    # TODO 5
    pass

TOOL_TICKET = {
    "name": "create_ticket",
    "description": ("Legt ein neues Störungsticket im Instandhaltungssystem an. Nur verwenden, wenn der Nutzer ausdrücklich "
                    "ein Ticket möchte. Prioritäten: hoch = Maschine steht oder Sicherheitsrisiko, mittel = Maschine läuft "
                    "eingeschränkt, niedrig = kein Einfluss auf die Produktion."),
    "parameters": {
        "type": "object",
        "properties": {
            "machine_id": {"type": "string", "description": "Maschinen-ID im Format M-01 bis M-08."},
            "description": {"type": "string", "description": "Kurze Beschreibung der Störung in einem Satz."},
            "priority": {"type": "string", "enum": ["hoch", "mittel", "niedrig"]},
        },
        "required": ["machine_id", "description", "priority"],
        "additionalProperties": False,
    },
}

In [ ]:
_vorher = len(TICKET_LOG)
check("TODO 5a Ticket wird angelegt", lambda: create_ticket("M-06", "Test", "niedrig")["status"] == "angelegt" and len(TICKET_LOG) == _vorher + 1)
check("TODO 5b Nummern steigen", lambda: TICKET_LOG[-1]["ticket"] == f"T-{1000 + len(TICKET_LOG)}")
check("TODO 5c ungültige Priorität wird abgelehnt", lambda: "Erlaubt" in (_wirft(create_ticket, "M-06", "Test", "dringend") or ""))
TICKET_LOG.clear()   # Testtickets wieder entfernen

## B3 · Eine generische Ausführung

### TODO 6 · `fuehre_tool_aus`
Schlagt die Funktion in `FUNKTIONEN` nach und ruft sie mit den Parametern aus `eingabe` auf (`funktion(**eingabe)`). Gebt `(ergebnis, False)` zurück. Geht **irgendetwas** schief, also ein unbekanntes Tool oder eine Ausnahme in der Funktion, gebt `({"fehler": <Meldung>}, True)` zurück. Der Assistent soll nie abstürzen, sondern dem Modell den Fehler melden.

In [ ]:
FUNKTIONEN = {"get_machine_status": get_machine_status, "get_maintenance_history": get_maintenance_history,
              "search_knowledge_base": search_knowledge_base, "create_ticket": create_ticket}
TOOLS = [TOOL_STATUS, TOOL_HISTORIE, TOOL_SUCHE, TOOL_TICKET]

def fuehre_tool_aus(name, eingabe):
    # TODO 6
    pass

In [ ]:
check("TODO 6a normaler Aufruf", lambda: fuehre_tool_aus("get_machine_status", {"machine_id": "M-01"})[1] is False)
check("TODO 6b unbekannte Maschine → Fehler statt Absturz", lambda: fuehre_tool_aus("get_machine_status", {"machine_id": "M-12"})[1] is True)
check("TODO 6c unbekanntes Tool → Fehler", lambda: fuehre_tool_aus("loesche_alles", {})[1] is True)
check("TODO 6d falscher Parameter → Fehler", lambda: fuehre_tool_aus("get_machine_status", {"maschine": "M-01"})[1] is True)

## B4 · Die Tool-Schleife

Das Modell kann mehrere Runden brauchen (erst suchen, dann Status abfragen …) und in einer Runde **mehrere Tools parallel** aufrufen. Die Schleife läuft, bis die Antwort keine `tool_calls` mehr enthält.

### TODO 7 · `frage_assistent`
In jeder Runde:
1. Das Modell aufrufen: System-Nachricht `SYSTEM_TOOLS` vorne im Verlauf, `tools=als_tools(TOOLS)`
2. Antwort mit `assistent_nachricht(...)` an den Verlauf anhängen
3. Enthält die Antwort keine `tool_calls`: Antworttext und Verlauf zurückgeben
4. Sonst: **jeden** Aufruf mit `ausfuehren(name, argumente)` ausführen (Argumente mit `json.loads` auspacken) und für **jeden** eine `tool`-Nachricht anhängen

In Woche 10 baut ihr daraus einen vollständigen Agenten mit Planung.

In [ ]:
SYSTEM_TOOLS = """Du bist der Instandhaltungs-Assistent der Nordwerk GmbH.
Nutze die Tools, statt Fakten zu raten. Fehlt eine nötige Angabe, etwa die Maschine, frage nach.
Nenne bei Informationen aus dem Handbuch die Quelle. Antworte knapp auf Deutsch."""

def frage_assistent(frage, ausfuehren=fuehre_tool_aus, max_runden=6, protokoll=True):
    nachrichten = [{"role": "system", "content": SYSTEM_TOOLS}, {"role": "user", "content": frage}]
    for runde in range(max_runden):
        # TODO 7
        pass
    return "Abbruch: zu viele Runden.", nachrichten

In [ ]:
erg = frage_assistent("Wann wurde an der Drehmaschine M-03 zuletzt das Spindellager geschmiert?")
TODO7_OK = not (erg[0].startswith("Abbruch") and len(erg[1]) == 2)
if not TODO7_OK:
    print("⏳ Erst TODO 7 lösen.")
else:
    print("\n💬", erg[0])
    check("TODO 7a Tool wurde benutzt", lambda: any(n["role"] == "tool" for n in erg[1]))
    check("TODO 7b Verlauf endet mit Antwort", lambda: erg[1][-1]["role"] == "assistant")

---
# Teil C · Schreibende Tools absichern

Lesende Tools sind harmlos. Schreibende Tools verändern die Welt: Tickets, E-Mails, Bestellungen. Hier gilt: **Der Mensch bestätigt, nicht das Modell.** Die Sicherung gehört in den Code, nicht in den Prompt, denn ein Prompt kann ignoriert oder ausgehebelt werden (Woche 13).

### TODO 8 · `fuehre_tool_aus_sicher`
Steht das Tool in `SCHREIBENDE_TOOLS`, zeigt `bestaetigen(name, eingabe)` die geplante Aktion an und fragt nach. Ohne Zustimmung wird **nicht** ausgeführt. Stattdessen geht `({"status": "abgelehnt", "hinweis": "Der Nutzer hat die Aktion abgelehnt."}, False)` zurück. Sonst wird normal `fuehre_tool_aus` aufgerufen.

In [ ]:
SCHREIBENDE_TOOLS = {"create_ticket"}

def frage_nutzer(name, eingabe):
    print(f"\n⚠️  Das Modell möchte ausführen: {name}({json.dumps(eingabe, ensure_ascii=False)})")
    return input("Ausführen? (j/n) ").strip().lower() in ("j", "ja")

def fuehre_tool_aus_sicher(name, eingabe, bestaetigen=frage_nutzer):
    # TODO 8
    pass

In [ ]:
_n = len(TICKET_LOG)
check("TODO 8a Ablehnung schreibt nichts", lambda: fuehre_tool_aus_sicher("create_ticket", {"machine_id": "M-06", "description": "x", "priority": "niedrig"}, bestaetigen=lambda *a: False)[0]["status"] == "abgelehnt" and len(TICKET_LOG) == _n)
check("TODO 8b Zustimmung schreibt", lambda: fuehre_tool_aus_sicher("create_ticket", {"machine_id": "M-06", "description": "x", "priority": "niedrig"}, bestaetigen=lambda *a: True)[0]["status"] == "angelegt" and len(TICKET_LOG) == _n + 1)
check("TODO 8c lesende Tools ohne Rückfrage", lambda: fuehre_tool_aus_sicher("get_machine_status", {"machine_id": "M-01"}, bestaetigen=lambda *a: 1 / 0)[1] is False)
TICKET_LOG.clear()

---
# Teil D · Ausprobieren und messen

Führt die Beispiele aus und beobachtet das Protokoll. Bei der Ticket-Anfrage werdet ihr nach Bestätigung gefragt.

In [ ]:
BEISPIELE = [
    "Vergleiche den Status von M-03 und M-05.",                                          # parallele Tool-Aufrufe?
    "Wie ist der Status von M-12?",                                                      # Fehlerfall
    "Die Drehmaschine vibriert. Was sagt das Handbuch, und wann wurde zuletzt geschmiert?",   # Suche plus Historie
    "Lege ein Ticket an: Lichtschranke an der Zuführung von M-06 verschmutzt, kein Einfluss auf die Produktion.",
    "Ist der Prüfstand noch elektrisch geprüft?",                                        # verlangt Kombination
]
for frage in (BEISPIELE if TODO7_OK and fuehre_tool_aus_sicher("get_machine_status", {"machine_id": "M-01"}) is not None else []):
    print(f"\n❓ {frage}")
    text, verlauf = frage_assistent(frage, ausfuehren=fuehre_tool_aus_sicher)
    print("💬", text)
if not (TODO7_OK and fuehre_tool_aus_sicher("get_machine_status", {"machine_id": "M-01"}) is not None):
    print("⏳ Erst TODO 7 und 8 lösen.")
print("\nAngelegte Tickets:", TICKET_LOG)

> **Zum Nachdenken:**
> - Hat das Modell bei der ersten Frage beide Maschinen in **einer** Runde abgefragt?
> - Wie ist es mit der unbekannten ID umgegangen?
> - Bei der letzten Frage muss das Modell zwei Dinge verknüpfen: die letzte Prüfung aus der Historie und das Intervall „jährlich“ aus dem Handbuch. Klappt das?

**Was kosten die Tools?** Jede Anfrage schickt alle Tool-Definitionen mit. Der Server baut sie in den Prompt ein, sie zählen also als Input-Tokens. Messen wir das über `usage` (Woche 4):

In [ ]:
try:
    _nachr = [{"role": "system", "content": SYSTEM_TOOLS}, {"role": "user", "content": "Status von M-03?"}]
    ohne = client.chat.completions.create(model=MODEL, max_tokens=1, messages=_nachr).usage.prompt_tokens
    mit = client.chat.completions.create(model=MODEL, max_tokens=1, messages=_nachr, tools=als_tools(TOOLS)).usage.prompt_tokens
    print(f"ohne Tools: {ohne} Tokens | mit 4 Tools: {mit} Tokens | Aufschlag: {mit - ohne} Tokens pro Anfrage und Runde")
except Exception as e:
    print("Übersprungen:", type(e).__name__)

---
## Bonus

**B1 · Die Beschreibung ist ein Prompt.** Kürzt die Beschreibung von `search_knowledge_base` auf „Sucht.“ und stellt die dritte Beispielfrage erneut. Benutzt das Modell das Tool noch richtig?

**B2 · Parallel oder nacheinander?** Setzt im Aufruf `parallel_tool_calls=False` und vergleicht die Zahl der Runden bei „Vergleiche den Status von M-03 und M-05.“ Achtung: Nicht jeder Dienst wertet diesen Parameter aus. Ändert sich nichts, ist auch das ein Ergebnis.

**B3 · Ein neues Tool.** Schreibt `list_open_tickets(machine_id=None)`, das offene Tickets aus `TICKETS` liefert, optional gefiltert. Ergänzt Funktion, Definition und `FUNKTIONEN`/`TOOLS`, und fragt: „Welche Tickets sind noch offen?“

In [ ]:
# B1 / B2 / B3

## Projekt diese Woche

Im Notebook **`W08_Projekt_Tools.ipynb`** baut ihr zwei bis drei Tools für euren Track plus `search_knowledge_base` um eure Wissensbasis aus Woche 7. Für beide Tracks gibt es Beispiel-Tools als Startpunkt. Mindestens ein Tool muss schreibend sein und abgesichert werden.

## Zusammenfassung

- **Das Modell schlägt vor, euer Code führt aus.** Der Ablauf: Definition, `tool_calls`, Ausführung, `tool`-Nachricht, Antwort.
- Die **Tool-Beschreibung ist ein Prompt**: was, wann, was kommt zurück, Abgrenzung zu anderen Tools.
- Ein sauberes Schema mit `required`, `enum` und `"additionalProperties": False` begrenzt erlaubte Parameter. Geprüft wird trotzdem im Code.
- Fehler werden als `tool`-Nachricht mit `FEHLER:` zurückgemeldet, mit einer Meldung, aus der das Modell lernen kann.
- **Schreibende Tools werden im Code abgesichert**, nicht im Prompt.
- Tool-Definitionen kosten bei jeder Anfrage Input-Tokens.

### Ausblick auf Woche 9
Wir haben jetzt viele Bausteine. Aber woher wissen wir, ob der Assistent gut ist, und ob eine Änderung ihn besser macht? Nächste Woche baut ihr ein **Evaluations-Harness**.